In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas seaborn matplotlib
from google.colab import drive
import os
import zipfile

# Mount Drive to access Data and Model Backup
drive.mount('/content/drive')

# Unzip Data (Required for Test Set)
zip_path = '/content/drive/MyDrive/hukom_data.zip' 
if os.path.exists(zip_path):
    print("Unzipping dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("✅ Data Ready.")
else:
    print("❌ Error: hukom_data.zip not found in Drive.")

In [ ]:
%%writefile hukom_utils.py
import pandas as pd
import os
from sklearn.model_selection import train_test_split

DATA_CSV = "hukom_dataset_final.csv"
CORPUS_DIR = "hukom_corpus_friendly"

def load_and_split_data():
    if not os.path.exists(DATA_CSV): return None, None, None
    df = pd.read_csv(DATA_CSV)
    df = df[df['label'] != -1]
    df['text_path'] = df['filename'].apply(lambda x: os.path.join(CORPUS_DIR, x))
    
    texts = []
    labels = []
    print("Reading text files...")
    for idx, row in df.iterrows():
        try:
            with open(row['text_path'], "r", encoding="utf-8") as f:
                content = f.read()
                if "========== FACTS ==========" in content:
                    parts = content.split("========== FACTS ==========")
                    if "========== RULING ==========" in parts[1]:
                        facts = parts[1].split("========== RULING ==========")[0].strip()
                        texts.append(facts)
                        labels.append(row['label'])
        except: continue

    train_txt, test_txt, train_lbl, test_lbl = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=42
    )
    # We only need the Test set for evaluation
    return None, None, (test_txt, test_lbl)

In [ ]:
%%writefile hukom_evaluator.py
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from hukom_utils import load_and_split_data
from torch.utils.data import Dataset

# --- CONFIGURATION ---
# OPTION A: Load from Hugging Face (If you pushed it)
# MODEL_PATH = "your-username/hukom-ai-legal-bert"

# OPTION B: Load from Google Drive Backup (If runtime crashed)
MODEL_PATH = "/content/drive/MyDrive/HukomAI/models/roberta_tagalog_headtail" 

MAX_LEN = 512

class HeadTailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        tokens = self.tokenizer(text, truncation=False, return_tensors='pt')
        input_ids = tokens['input_ids'][0]
        
        if len(input_ids) > self.max_len:
            head = input_ids[:129]
            tail = input_ids[-(self.max_len - 129):]
            input_ids = torch.cat([head, tail])
        else:
            padding = torch.zeros(self.max_len - len(input_ids), dtype=torch.long)
            input_ids = torch.cat([input_ids, padding])

        return {
            'input_ids': input_ids,
            'attention_mask': (input_ids != 0).long(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def run_evaluation():
    print(f"Loading model from: {MODEL_PATH}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("Check if the path in MODEL_PATH is correct!")
        return

    print("Loading Test Data...")
    _, _, (test_txt, test_lbl) = load_and_split_data()
    
    test_dataset = HeadTailDataset(test_txt, test_lbl, tokenizer)
    
    print("Running Predictions...")
    trainer = Trainer(model=model)
    preds_output = trainer.predict(test_dataset)
    y_preds = preds_output.predictions.argmax(-1)
    y_true = preds_output.label_ids

    target_names = ["Acquittal (0)", "Conviction (1)", "Modification (2)", "Remand (3)"]
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(y_true, y_preds, target_names=target_names, digits=4))

    cm = confusion_matrix(y_true, y_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.ylabel('Actual Verdict')
    plt.xlabel('Predicted Verdict')
    plt.title('Hukom-AI Confusion Matrix')
    plt.show()

if __name__ == "__main__":
    run_evaluation()

In [ ]:
!python hukom_evaluator.py